# Scientific Validity of PPI Mean Monitoring

Once a GenAI system is deployed, the question shifts from "what is the metric today?" to "has the metric drifted?" GLIDE's monitors re-estimate a metric on successive batches of production data and raise an alarm the moment there is statistically valid evidence that it has crossed a threshold. Unlike the one-off estimators validated elsewhere in this section, a monitor is not judged by interval coverage of a fixed estimand: it is judged by its **false-alarm probability under no drift** and by its **detection power and speed under actual drift**. This notebook empirically verifies both properties for **Prediction-Powered Risk Monitoring (PPRM)**.

Throughout this notebook, the metric is treated as a **risk** (lower is better).

We compare three methods, run on every scenario below, each dropping one virtue of PPRM to isolate its effect: **PPRM** (`PPIMeanMonitor`) targets the debiased true mean, is anytime-valid, and uses the proxy; **Classical** (`ClassicalMeanMonitor`) also targets a true-risk estimand and is anytime-valid, but drops the proxy; **Naive peeking** (`PPIMeanEstimator` per batch) targets the debiased true mean as well but drops anytime-validity. It is statistically invalid by construction and is included for comparison only.


In [ ]:
import matplotlib.pyplot as plt
import numpy as np

from glide.estimators import ClassicalMeanEstimator, PPIMeanEstimator
from glide.monitors import ClassicalMeanMonitor, PPIMeanMonitor
from glide.samplers import StratifiedSampler
from glide.scientific_validation import coverage_with_error_bar
from glide.simulators import generate_stratified_binary_dataset, simulate_annotation

plt.rcParams.update(
    {
        "font.size": 18,
        "axes.labelsize": 18,
        "axes.titlesize": 18,
        "legend.fontsize": 16,
        "xtick.labelsize": 16,
        "ytick.labelsize": 16,
        "figure.titlesize": 19,
    }
)

## Experiment Parameters

We fix every parameter up front. Each stratum returned by `generate_stratified_binary_dataset` is read as one chronologically monitored batch: the `groups` array it returns is used directly as `batches`, and every batch is monitored.

- `N_BATCHES`, `BATCH_SIZE`, `LABELS_PER_BATCH`: stream length, number of samples per batch, and human-labeled samples per batch (allocated with `StratifiedSampler`).
- `TRUE_MEAN`, `PROXY_BIAS`: the stationary true risk and the (fixed) proxy bias, so `proxy_mean = true_mean + PROXY_BIAS` at every batch, before and after any drift. The proxy always co-moves with the true risk.
- `CORRELATION`: the proxy/true correlation value, held fixed and reflecting an LLM-judge's quality.
- `THRESHOLD`: the alarm boundary, the highest tolerated risk value.

- `DRIFT_START`, `AMPLITUDES`: for the abrupt-drift workflow only, the batch at which `true_mean` and `proxy_mean` step up together, and the grid of step sizes swept in the detection-power figure.
- `DEFAULT_CONFIDENCE_LEVEL`, `CONFIDENCE_LEVELS`: the confidence level used throughout, and the grid swept in the calibration figure (`1 - confidence_level` is the total false-alarm budget).
- `N_SEEDS`, `ERROR_BAR_CONFIDENCE_LEVEL`: the number of Monte Carlo streams, and the confidence level of the error bars drawn on every estimated rate or delay.


In [ ]:
N_BATCHES = 200
BATCH_SIZE = 160
LABELS_PER_BATCH = 40
TRUE_MEAN = 0.3
PROXY_BIAS = 0.05
CORRELATION = 0.85
THRESHOLD = 0.35

DRIFT_START = 25
AMPLITUDES = np.array([0.1, 0.15, 0.2, 0.25, 0.3])

DEFAULT_CONFIDENCE_LEVEL = 0.8
CONFIDENCE_LEVELS = np.array([0.5, 0.6, 0.7, 0.8, 0.9])

N_SEEDS = 250
ERROR_BAR_CONFIDENCE_LEVEL = 0.9

METHODS = ["PPRM", "Classical", "Naive (peeking)"]
VALID_METHODS = ["PPRM", "Classical"]
METHOD_COLORS = {"PPRM": "darkorange", "Classical": "steelblue", "Naive (peeking)": "red"}

## Data generation

The functions in the next cell generate simulated data for two scenarios: 

- No drift: all data batches are drawn containing true and proxy labels following Bernoulli distributions with a constant mean

- Abrupt drift: the initial batches follow a non alarming distribution up to `DRIFT_START` and the ones thereafter having a different mean risk value above the alarm threshold.

True and proxy labels are initially generated for all instances. Then a fixed number of true labels are kept within each batch and the rest masked to simulate what would be observed in practice.

In [ ]:
def simulate_no_drift_stream(seed):
    true_means = np.full(N_BATCHES, TRUE_MEAN)
    proxy_means = true_means + PROXY_BIAS
    correlations = np.full(N_BATCHES, CORRELATION)
    y_true_oracle, y_proxy, batches = generate_stratified_binary_dataset(
        n_total=np.full(N_BATCHES, BATCH_SIZE),
        true_mean=true_means,
        proxy_mean=proxy_means,
        correlation=correlations,
        random_seed=seed,
    )
    xi = StratifiedSampler().sample(
        y_proxy, groups=batches, n_samples=LABELS_PER_BATCH * N_BATCHES, strategy="proportional", random_seed=seed
    )
    y_true = simulate_annotation(y_true_oracle, xi)
    return y_true, y_proxy, batches


def simulate_abrupt_drift_stream(seed, amplitude):
    true_means = np.full(N_BATCHES, TRUE_MEAN)
    true_means[DRIFT_START:] += amplitude
    proxy_means = true_means + PROXY_BIAS
    correlations = np.full(N_BATCHES, CORRELATION)
    y_true_oracle, y_proxy, batches = generate_stratified_binary_dataset(
        n_total=np.full(N_BATCHES, BATCH_SIZE),
        true_mean=true_means,
        proxy_mean=proxy_means,
        correlation=correlations,
        random_seed=seed,
    )
    xi = StratifiedSampler().sample(
        y_proxy, groups=batches, n_samples=LABELS_PER_BATCH * N_BATCHES, strategy="proportional", random_seed=seed
    )
    y_true = simulate_annotation(y_true_oracle, xi)
    return y_true, y_proxy, batches

The functions below implement each of the three monitoring methods. The naive baseline calls `PPIMeanEstimator` on each batch in isolation and alarms the moment a single batch's confidence interval lies entirely above the threshold, this is the repeated-testing procedure described in the user guide.

The `run_all_methods` takes care of running all methods for one instance of generated data and returning their first alarm indices. This is all we need for the following analysis on false alarm rate, detection rate and detection delay.

In [ ]:
def naive_first_alarm_index(y_true, y_proxy, batches, confidence_level, threshold):
    n_batches = len(np.unique(batches))
    for batch_id in range(n_batches):
        batch_mask = batches == batch_id
        batch_result = PPIMeanEstimator().estimate(
            y_true[batch_mask], y_proxy[batch_mask], confidence_level=confidence_level
        )
        if batch_result.confidence_interval.lower_bound > threshold:
            return batch_id
    return n_batches


def run_all_methods(y_true, y_proxy, batches, confidence_level, threshold):
    pprm_result = PPIMeanMonitor().detect(
        y_true, y_proxy, batches, higher_is_better=False, threshold=threshold, confidence_level=confidence_level
    )
    classical_result = ClassicalMeanMonitor().detect(
        y_true, batches, higher_is_better=False, threshold=threshold, confidence_level=confidence_level
    )
    naive_index = naive_first_alarm_index(y_true, y_proxy, batches, confidence_level, threshold)

    pprm_index = pprm_result.first_alarm_index if pprm_result.drift_detected else N_BATCHES
    classical_index = classical_result.first_alarm_index if classical_result.drift_detected else N_BATCHES
    return {"PPRM": pprm_index, "Classical": classical_index, "Naive (peeking)": naive_index}

A stream that never alarms is given a sentinel `first_alarm_index` of `N_BATCHES` (one past the last valid index), so the indicator "alarmed by batch `t`" (`first_alarm_index <= t`) is `False` at every `t` for a stream that never alarms, without a separate `None` case to track.

### A single stream, for intuition

Before the Monte Carlo experiments, one simulated abrupt-drift stream shows what `detect` actually returns: a running mean and an anytime-valid bound on it, batch by batch.


In [ ]:
AMPLITUDE_EXAMPLE = 0.15
y_true_example, y_proxy_example, batches_example = simulate_abrupt_drift_stream(seed=1, amplitude=AMPLITUDE_EXAMPLE)
pprm_example = PPIMeanMonitor().detect(
    y_true_example,
    y_proxy_example,
    batches_example,
    higher_is_better=False,
    threshold=THRESHOLD,
    confidence_level=DEFAULT_CONFIDENCE_LEVEL,
    max_tuning_parameter=0.1,
)
classical_example = ClassicalMeanMonitor().detect(
    y_true_example,
    batches_example,
    higher_is_better=False,
    threshold=THRESHOLD,
    confidence_level=DEFAULT_CONFIDENCE_LEVEL,
)

fig, ax = plt.subplots(figsize=(9, 5))
batch_axis = np.arange(1, N_BATCHES + 1)
for name, result in [("PPRM", pprm_example), ("Classical", classical_example)]:
    color = METHOD_COLORS[name]
    ax.plot(batch_axis, result.confidence_bounds, color=color, label=name)

    exceeds_threshold = np.where(result.confidence_bounds > THRESHOLD)[0]
    if exceeds_threshold.size > 0:
        first_exceed_batch = batch_axis[exceeds_threshold[0]]
        ax.axvline(first_exceed_batch, color=color, linestyle=":", lw=1.5)

true_risk_curve = np.where(batch_axis < DRIFT_START, TRUE_MEAN, TRUE_MEAN + AMPLITUDE_EXAMPLE)
ax.plot(batch_axis, true_risk_curve, color="dimgray", linestyle="-.", lw=1.5, label="True risk")
ax.axhline(THRESHOLD, color="black", linestyle="--", lw=1.5, label="Threshold")
ax.axvline(DRIFT_START, color="gray", linestyle=":", lw=1.5, label="Drift start")
ax.set_xlabel("Batches")
ax.set_ylabel("Risk lower bound")
ax.legend()
plt.tight_layout()
plt.show()

The bound climbs after the drift starts and eventually crosses the threshold, at which point `detect` reports `drift_detected=True`. The Monte Carlo experiments below repeat this once per seed, at scale, to characterize false-alarm and detection behavior across the whole distribution of streams rather than a single draw.


## The Monte Carlo Loop

A single external loop over `range(N_SEEDS)` calls both workflows once per seed. For the no-drift workflow, the same simulated stream is reused across the `CONFIDENCE_LEVELS` grid. For the abrupt-drift workflow, each amplitude in `AMPLITUDES` requires its own stream, so data is regenerated per amplitude at the fixed `DEFAULT_CONFIDENCE_LEVEL`. The two workflows do not share data with each other.


In [ ]:
no_drift_first_alarms = {
    level: {method: np.empty(N_SEEDS, dtype=int) for method in METHODS} for level in CONFIDENCE_LEVELS
}
abrupt_drift_first_alarms = {
    amplitude: {method: np.empty(N_SEEDS, dtype=int) for method in METHODS} for amplitude in AMPLITUDES
}

for seed in range(N_SEEDS):
    y_true, y_proxy, batches = simulate_no_drift_stream(seed)
    for level in CONFIDENCE_LEVELS:
        outcomes = run_all_methods(y_true, y_proxy, batches, confidence_level=level, threshold=THRESHOLD)
        for method in METHODS:
            no_drift_first_alarms[level][method][seed] = outcomes[method]

    for amplitude in AMPLITUDES:
        y_true, y_proxy, batches = simulate_abrupt_drift_stream(seed, amplitude)
        outcomes = run_all_methods(
            y_true, y_proxy, batches, confidence_level=DEFAULT_CONFIDENCE_LEVEL, threshold=THRESHOLD
        )
        for method in METHODS:
            abrupt_drift_first_alarms[amplitude][method][seed] = outcomes[method]

## False-Alarm Control Under No Drift

The stream is stationary: all batches are sampled from the same distribution whose risk is below the threshold, so no method should alarm except by chance. Checking a fresh confidence interval after every batch, as the naive baseline does, is a repeated-testing (peeking) failure: even though every individual look was calibrated, the repeated tests are almost sure to raise a false alarm at long horizon. An anytime-valid confidence sequence, in contrast, bounds the probability of *ever* alarming over the whole horizon by the single budget.

### Cumulative false-alarm rate

For each batch index `t`, the plot below shows the empirical probability that a stream has alarmed by `t`: the fraction of the `N_SEEDS` streams whose `first_alarm_index <= t - 1`, computed at `DEFAULT_CONFIDENCE_LEVEL`. Because a stream that has alarmed stays alarmed. This is the empirical CDF of `first_alarm_index` and is non-decreasing in `t`.


In [ ]:
batch_axis = np.arange(1, N_BATCHES + 1)
default_level_alarms = no_drift_first_alarms[DEFAULT_CONFIDENCE_LEVEL]
budget = 1 - DEFAULT_CONFIDENCE_LEVEL

fig, ax = plt.subplots(figsize=(9, 5))
for method in METHODS:
    first_alarms = default_level_alarms[method]
    means, lowers, uppers = np.zeros(len(batch_axis)), np.zeros(len(batch_axis)), np.zeros(len(batch_axis))
    for i in range(N_BATCHES):
        hits = (first_alarms <= batch_axis[i] - 1).astype(float)
        mean_rate, lower, upper = coverage_with_error_bar(hits, ERROR_BAR_CONFIDENCE_LEVEL)
        means[i] = mean_rate
        lowers[i] = lower
        uppers[i] = upper
    color = METHOD_COLORS[method]
    ax.plot(batch_axis, means, color=color, label=method)
    ax.fill_between(batch_axis, lowers, uppers, alpha=0.15, color=color)

ax.axhline(budget, color="black", linestyle="--", lw=2, label=f"Budget (1 - confidence level = {budget:.2f})")
ax.set_xlabel("Batches observed")
ax.set_ylabel("Alarm rate")
ax.legend()
plt.tight_layout()
plt.show()

The naive baseline's curve climbs steadily as more batches accumulate more chances to false-alarm, consistent with `1 - (1 - alpha)^t`. It crosses the budget line before the end of the horizon: peeking is unsafe. PPRM and Classical both stay flat and near zero for every `t`, including the terminal one. Their anytime-valid guarantee bounds the probability of *ever* alarming over the whole horizon. The empirical-Bernstein construction that provides it is conservative, so the observed rate typically sits well below the nominal budget.

### Calibration: terminal false-alarm rate versus budget

The figure above fixes `confidence_level`. The calibration figure below sweeps `CONFIDENCE_LEVELS` and plots each method's **terminal** false-alarm rate (the proportion of streams that ever alarm over the whole horizon) against the budget `alpha = 1 - confidence_level`, the monitoring analog of the coverage-versus-nominal-level check used for the estimators. The same simulated streams are reused at every level: only `detect` is re-run, since `confidence_level` is the only input that changes.


In [ ]:
alphas = 1 - CONFIDENCE_LEVELS

fig, ax = plt.subplots(figsize=(8, 5))
ax.plot(alphas, alphas, color="black", linestyle="--", lw=1.5, label="Budget (y = alpha)")
for method in METHODS:
    means, lowers, uppers = [], [], []
    for level in CONFIDENCE_LEVELS:
        first_alarms = no_drift_first_alarms[level][method]
        hits = (first_alarms < N_BATCHES).astype(float)
        mean_rate, lower, upper = coverage_with_error_bar(hits, ERROR_BAR_CONFIDENCE_LEVEL)
        means.append(mean_rate)
        lowers.append(lower)
        uppers.append(upper)
    color = METHOD_COLORS[method]
    ax.plot(alphas, means, marker="o", color=color, label=method)
    ax.fill_between(alphas, lowers, uppers, alpha=0.15, color=color)

ax.set_xlabel("False-alarm budget (alpha = 1 - confidence level)")
ax.set_ylabel("Terminal false-alarm rate")
ax.legend()
plt.tight_layout()
plt.show()

At every budget tested, the naive peeking baseline sits above the `y = alpha` diagonal, exceeding the very budget it was configured with. PPRM and Classical both sit on or below the diagonal at every level, generalizing the single-budget comparison above into a full calibration statement: peeking invalidates the false-alarm guarantee at any confidence level, while the anytime-valid construction respects it at all of them.

---

## Detection Power and Speed Under Abrupt Drift


The stream now steps at batch `DRIFT_START`: `true_mean` and `proxy_mean` co-move from their stationary values to `TRUE_MEAN + amplitude` (and `TRUE_MEAN + amplitude + PROXY_BIAS`), at the same fixed `CORRELATION`, for the remaining batches. Because PPRM's estimand is the debiased true mean, not the proxy mean, only a genuine shift in `true_mean` constitutes drift; the proxy is kept co-moving purely so it stays informative throughout, not because the proxy mean itself is what is being monitored. `AMPLITUDES` sweeps the step size, and for each (amplitude, seed) pair we record whether each method ever alarms and, if so, at what delay.

Only PPRM and Classical are compared here: a delay axis is not meaningful for the naive baseline, which alarms early for the wrong reason (peeking) rather than because of genuine evidence, and whose invalidity is already established above.


In [ ]:
fig, (ax_rate, ax_delay) = plt.subplots(1, 2, figsize=(14, 6))
for method in VALID_METHODS:
    color = METHOD_COLORS[method]
    plotted_amplitudes, rate_means, rate_lowers, rate_uppers = [], [], [], []
    delay_means, delay_lowers, delay_uppers = [], [], []
    for amplitude in AMPLITUDES:
        first_alarms = abrupt_drift_first_alarms[amplitude][method]
        detection_mask = first_alarms < N_BATCHES
        detected = first_alarms[detection_mask]
        if sum(detection_mask) < 2:
            continue
        detection_rate, rate_lower, rate_upper = coverage_with_error_bar(
            detection_mask.astype(float), ERROR_BAR_CONFIDENCE_LEVEL
        )
        delay_result = ClassicalMeanEstimator().estimate(
            detected.astype(float), confidence_level=ERROR_BAR_CONFIDENCE_LEVEL
        )
        plotted_amplitudes.append(amplitude)
        rate_means.append(detection_rate)
        rate_lowers.append(rate_lower)
        rate_uppers.append(rate_upper)
        delay_means.append(delay_result.mean)
        delay_lowers.append(delay_result.confidence_interval.lower_bound)
        delay_uppers.append(delay_result.confidence_interval.upper_bound)

    ax_rate.plot(plotted_amplitudes, rate_means, marker="o", color=color, label=method)
    ax_rate.fill_between(plotted_amplitudes, rate_lowers, rate_uppers, alpha=0.15, color=color)

    ax_delay.plot(plotted_amplitudes, delay_means, marker="o", color=color, label=method)
    ax_delay.fill_between(plotted_amplitudes, delay_lowers, delay_uppers, alpha=0.15, color=color)

ax_rate.set_xlabel("Amplitude")
ax_rate.set_ylabel("Detection rate")
ax_rate.legend()

ax_delay.set_xlabel("Amplitude")
ax_delay.set_ylabel("Mean detection delay")
ax_delay.legend()

plt.tight_layout()
plt.show()

The detection rate is plotted against amplitude on the left panel. The right panel plots the mean detection delay per detection method against amplitude. As `amplitude` grows, detection rate rises and mean detection delay falls for both monitors. We can see that PPRM has better detection rate and shorter detection delays compare to the classical method.

## Summary

| Method | Valid under repeated looks? | Detects abrupt drift? | Detection delay, relative to Classical |
|---|---|---|---|
| **PPRM** | Yes | Yes | Shorter |
| **Classical** | Yes | Yes | Baseline |
| **Naive peeking** | No | Yes, but invalidly | Not comparable (invalid) |

Crucially, Naive peeking is invalid for continuous risk monitoring and inevitably exceeds the budget at long horizons. On the other hand, both `PPIMeanMonitor` (PPRM) and `ClassicalMeanMonitor` control the false-alarm rate at every confidence level. Moreover, we have seen that PPRM leverages proxy labels to improve detection rate and reduce detection delay.